# Amazon Food Review Classification using LSTM

Classify food reviews into score categories (1-5) using an LSTM model on 10% of the dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

## 1. Load and Sample Data

In [ ]:
df = pd.read_csv('amazon_review.csv')
print(f'Full dataset shape: {df.shape}')
print(f'\nScore distribution (full):')
print(df['Score'].value_counts().sort_index())
df.head()

In [ ]:
df_sampled = df.sample(frac=0.10, random_state=42)
print(f'Sampled dataset shape: {df_sampled.shape}')
print(f'\nScore distribution (sampled):')
print(df_sampled['Score'].value_counts().sort_index())

In [ ]:
df_sampled = df_sampled.dropna(subset=['Text', 'Score'])
df_sampled['Score'] = df_sampled['Score'].astype(int)

plt.figure(figsize=(8, 4))
df_sampled['Score'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Score Distribution (10% Sample)')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 64

texts = df_sampled['Text'].values
labels = (df_sampled['Score'] - 1).values  # shift to 0-indexed
num_classes = len(np.unique(labels))

print(f'Number of classes: {num_classes}')
print(f'Class names: {np.unique(labels) + 1}')

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
print(f'Unique tokens: {len(word_index)}')

X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
y = to_categorical(labels, num_classes=num_classes)

print(f'Input shape: {X.shape}')
print(f'Output shape: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

from sklearn.utils.class_weight import compute_class_weight

y_train_labels = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train_labels)
class_weights = dict(enumerate(class_weights_array))

print(f'\nClass weights (to handle imbalance):')
for i, w in enumerate(class_weights):
    print(f'  Score {i+1}: {class_weights[i]:.3f}')

## 2b. Text Augmentation for Class Balance

In [ ]:
import random
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import wordnet

def get_synonyms(word):
    synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name().lower() != word.lower():
                synonyms.append(lemma.name().replace('_', ' '))
    return list(set(synonyms))

def augment_synonym(text, p=0.25):
    words = text.split()
    return ' '.join(
        random.choice(get_synonyms(w)) if random.random() < p and get_synonyms(w) else w
        for w in words
    )

def augment_swap(text, n_swaps=3):
    words = text.split()
    if len(words) < 2:
        return text
    for _ in range(min(n_swaps, len(words)//4)):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

def augment_delete(text, p=0.2):
    words = text.split()
    if len(words) <= 3:
        return text
    return ' '.join(w for w in words if random.random() > p)

def augment_insert(text, n=3):
    words = text.split()
    new_words = list(words)
    for _ in range(n):
        idx = random.randint(0, len(new_words))
        word = random.choice(new_words) if new_words else ''
        syns = get_synonyms(word) if word else []
        if syns:
            new_words.insert(idx, random.choice(syns))
    return ' '.join(new_words)

def augment_repeat(text, n=2):
    words = text.split()
    if len(words) < 2:
        return text
    for _ in range(n):
        idx = random.randint(0, len(words)-1)
        words.insert(idx+1, words[idx])
    return ' '.join(words)

AUGMENTERS = [augment_synonym, augment_swap, augment_delete, augment_insert, augment_repeat]

def augment_text(text, n_augments=3):
    return [random.choice(AUGMENTERS)(text) for _ in range(n_augments)]

print('Augmenters ready: 5 methods (synonym, swap, delete, insert, repeat)')

In [ ]:
TARGET_PER_CLASS = 3000

augmented_texts = []
augmented_labels = []

print(f'Target samples per class: {TARGET_PER_CLASS}')
print(f'\nBalancing dataset...')

for cls in range(num_classes):
    cls_texts = df_sampled[df_sampled['Score'] == cls + 1]['Text'].dropna().tolist()
    count = len(cls_texts)
    
    if count >= TARGET_PER_CLASS:
        selected = random.sample(cls_texts, TARGET_PER_CLASS)
        augmented_texts.extend(selected)
        augmented_labels.extend([cls] * TARGET_PER_CLASS)
        print(f'  Score {cls+1}: {count} -> undersampled to {TARGET_PER_CLASS}')
    else:
        deficit = TARGET_PER_CLASS - count
        augmented_texts.extend(cls_texts)
        augmented_labels.extend([cls] * count)
        n_aug = max(2, deficit // count + 1)
        aug_count = 0
        for text in cls_texts:
            if aug_count >= deficit:
                break
            aug_texts = augment_text(text, n_augments=n_aug)
            for at in aug_texts:
                if aug_count >= deficit:
                    break
                augmented_texts.append(at)
                augmented_labels.append(cls)
                aug_count += 1
        print(f'  Score {cls+1}: {count} -> augmented to {count + aug_count}')

print(f'\nTotal balanced samples: {len(augmented_texts)}')

In [ ]:
aug_sequences = tokenizer.texts_to_sequences(augmented_texts)
X_aug = pad_sequences(aug_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
y_aug = to_categorical(augmented_labels, num_classes=num_classes)

shuffle_idx = np.random.permutation(len(X_aug))
X_aug = X_aug[shuffle_idx]
y_aug = y_aug[shuffle_idx]

X_train_final = X_aug[:int(0.85 * len(X_aug))]
y_train_final = y_aug[:int(0.85 * len(y_aug))]
X_val = X_aug[int(0.85 * len(X_aug)):]
y_val = y_aug[int(0.85 * len(y_aug)):]

print(f'Final train: {X_train_final.shape[0]} samples')
print(f'Validation: {X_val.shape[0]} samples')
print(f'Test (held out): {X_test.shape[0]} samples')

new_counts = Counter(np.argmax(y_train_final, axis=1))
print(f'\nBalanced train distribution:')
for cls in sorted(new_counts.keys()):
    print(f'  Score {cls+1}: {new_counts[cls]}')

## 3. Build LSTM Model

In [ ]:
from tensorflow.keras.layers import LayerNormalization, GlobalAveragePooling1D

model = Sequential([
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    SpatialDropout1D(0.4),
    Bidirectional(LSTM(128, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.4),
    Bidirectional(LSTM(64, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.4),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_final, y_train_final,
    epochs=15,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Evaluation - Precision and Recall per Category

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

score_names = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']

print('=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_true_classes, y_pred_classes, target_names=score_names, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_classes, y_pred_classes, average=None
)

metrics_df = pd.DataFrame({
    'Score': score_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print('\n' + '=' * 60)
print('PRECISION & RECALL PER CATEGORY')
print('=' * 60)
print(metrics_df.to_string(index=False))

print(f'\nMacro Average Precision: {np.mean(precision):.4f}')
print(f'Macro Average Recall: {np.mean(recall):.4f}')
print(f'Weighted Average Precision: {np.average(precision, weights=support):.4f}')
print(f'Weighted Average Recall: {np.average(recall, weights=support):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(score_names))
width = 0.35

axes[0].bar(x_pos - width/2, precision, width, label='Precision', color='steelblue', edgecolor='black')
axes[0].bar(x_pos + width/2, recall, width, label='Recall', color='coral', edgecolor='black')
axes[0].set_xlabel('Score Category')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision & Recall per Category')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(score_names)
axes[0].legend()
axes[0].set_ylim(0, 1)

cm = confusion_matrix(y_true_classes, y_pred_classes)
im = axes[1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Confusion Matrix')
plt.colorbar(im, ax=axes[1])
tick_marks = np.arange(num_classes)
axes[1].set_xticks(tick_marks)
axes[1].set_xticklabels(score_names, rotation=45)
axes[1].set_yticks(tick_marks)
axes[1].set_yticklabels(score_names)
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

fmt = 'd'
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, format(cm[i, j], fmt),
                     ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()

## 6. Overall Accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')